In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Question 1:

In [90]:
df_orders= pd.read_csv('../dataset/orders.csv', parse_dates=['order_date'])

In [91]:
df_orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [92]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  object        
 5   payment_method  646945 non-null  object        
 6   device_type     646945 non-null  object        
 7   order_source    646945 non-null  object        
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 39.5+ MB


In [93]:
customer_count= df_orders.groupby('customer_id').size()
multi_order_customers = customer_count[customer_count >= 2]
print(f"Số lượng khách hàng đã đặt hàng nhiều lần: {len(multi_order_customers)}")

Số lượng khách hàng đã đặt hàng nhiều lần: 67888


In [94]:
orders_multi= df_orders[df_orders['customer_id'].isin(multi_order_customers.index)]
orders_multi= orders_multi.sort_values(by=['customer_id', 'order_date'])
orders_multi['prev_date']= orders_multi.groupby('customer_id')['order_date'].shift(1)
orders_multi['days_between']= (orders_multi['order_date'] - orders_multi['prev_date']).dt.days
median_between= orders_multi['days_between'].median()
print(f"Thời gian trung bình giữa các đơn hàng của khách hàng: {median_between:.2f} ngày") 

Thời gian trung bình giữa các đơn hàng của khách hàng: 144.00 ngày


### ==> Đáp án: C

## Question 2:

In [95]:
df_products= pd.read_csv('../dataset/products.csv')

In [96]:
df_products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [97]:
df_products['segment'].unique()

array(['Everyday', 'Performance', 'Balanced', 'Standard', 'All-weather',
       'Premium', 'Trendy', 'Activewear'], dtype=object)

In [98]:
gross_margin= (df_products['price']- df_products['cogs'])/ df_products['price']
df_products['gross_margin'] = gross_margin
segment_margin= df_products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print("Lợi nhuận gộp trung bình theo phân khúc sản phẩm:")
print(segment_margin)

Lợi nhuận gộp trung bình theo phân khúc sản phẩm:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


### ==> Đáp án: D

## Question 3:

In [99]:
df_returns= pd.read_csv('../dataset/returns.csv')
df_merge= pd.merge(df_products, df_returns, on='product_id', how='left')

In [100]:
df_merge.head()

,product_id,product_name,category,segment,size,color,price,cogs,gross_margin,return_id,order_id,return_date,return_reason,return_quantity,refund_amount
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225,RET-017121,268094.0,2015-04-21,not_as_described,2.0,17631.46
1,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225,RET-017720,277786.0,2015-05-19,wrong_size,8.0,87302.64
2,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336,RET-034568,549673.0,2017-12-20,changed_mind,1.0,6580.77
3,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336,RET-035665,566897.0,2018-03-31,wrong_size,1.0,8969.16
4,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336,RET-037079,590682.0,2018-06-12,defective,5.0,46022.01


In [101]:
df_streetwear= df_merge[df_merge['category'] == 'Streetwear']
most_return_reason= df_streetwear.groupby('return_reason').size().sort_values(ascending=False)
print("Lý do trả hàng phổ biến nhất của Streetwear:", most_return_reason.index[0])

Lý do trả hàng phổ biến nhất của Streetwear: wrong_size


### ==> Đáp án: B

## Question 4:

In [102]:
df_web_traffic= pd.read_csv('../dataset/web_traffic.csv', parse_dates=['date'])

In [103]:
df_web_traffic.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


In [104]:
print(df_web_traffic['traffic_source'].unique())

['organic_search' 'direct' 'referral' 'social_media' 'paid_search'
 'email_campaign']


In [105]:
avg_bound_rate= df_web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values(ascending=True)
print("Tỷ lệ thoát trung bình theo nguồn truy cập:")
print(avg_bound_rate)

Tỷ lệ thoát trung bình theo nguồn truy cập:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


### ==> Đáp án: C

## Question 5:

In [106]:
df_order_items= pd.read_csv('../dataset/order_items.csv', low_memory=False)

In [107]:
df_order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [108]:
total_row= len(df_order_items)
print(f"Tổng số dòng trong order_items: {total_row}")

Tổng số dòng trong order_items: 714669


In [109]:
is_promo= df_order_items['promo_id'].notna() | df_order_items['promo_id_2'].notna()
print(f"Số lượng dòng có promo_id hoặc promo_id_2 không rỗng:", len(df_order_items[is_promo]))

Số lượng dòng có promo_id hoặc promo_id_2 không rỗng: 276316


In [110]:
print("Tỷ lệ phần trăm áp dụng khuyến mãi:", len(df_order_items[is_promo]) / total_row * 100)

Tỷ lệ phần trăm áp dụng khuyến mãi: 38.663493169565214


### ==> Đáp án: C

## Question 6:

In [111]:
df_customers= pd.read_csv('../dataset/customers.csv')

In [112]:
df_customers.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


In [113]:
df=pd.merge(df_orders, df_customers, on='customer_id', how='left')
df.head()

,order_id,order_date,customer_id,zip_x,order_status,payment_method,device_type,order_source,zip_y,city,signup_date,gender,age_group,acquisition_channel
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,1109,Hanoi,2020-06-06,Female,35-44,social_media
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1330,Phu Ly,2021-11-03,Female,18-24,social_media
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,1473,Lao Cai,2020-09-18,Female,35-44,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,2360,Son Tay,2016-05-29,Male,45-54,direct
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,2886,Uong Bi,2017-07-11,Male,18-24,social_media


In [114]:
mean_group= df.groupby('age_group')['order_id'].count() / df.groupby('age_group')['customer_id'].nunique()
print("Số đơn hàng trung bình mỗi khách hàng theo nhóm tuổi:")
print(mean_group)

Số đơn hàng trung bình mỗi khách hàng theo nhóm tuổi:
age_group
18-24    7.068577
25-34    7.112230
35-44    7.206159
45-54    7.220264
55+      7.268731
dtype: float64


### ==> Đáp án: A

## Question 7

In [115]:
df_geo= pd.read_csv('../dataset/geography.csv')

In [116]:
df_geo.head()

,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13


In [117]:
df_geo['region'].unique()

array(['East', 'Central', 'West'], dtype=object)

In [118]:
df_q7= df_orders.copy()

In [119]:
df_merge= pd.merge(df_geo, df_q7, on='zip', how='left')
df_merge.head()

,zip,city,region,district,order_id,order_date,customer_id,order_status,payment_method,device_type,order_source
0,15201,Hai Phong,East,District #13,5280.0,2012-07-25,1.0,delivered,cod,desktop,paid_search
1,15201,Hai Phong,East,District #13,9294.0,2012-08-09,5.0,delivered,credit_card,desktop,paid_search
2,15201,Hai Phong,East,District #13,13047.0,2012-08-27,3.0,delivered,credit_card,mobile,social_media
3,15201,Hai Phong,East,District #13,31032.0,2012-11-26,5.0,delivered,cod,tablet,social_media
4,15201,Hai Phong,East,District #13,31033.0,2012-11-24,3.0,delivered,apple_pay,desktop,social_media


In [120]:
df_final_merge= pd.merge(df_merge, df_order_items, on='order_id', how='left')
df_final_merge['Revenue']= df_final_merge['quantity'] * df_final_merge['unit_price'] - df_final_merge['discount_amount']
df_final_merge.head()

,zip,city,region,district,order_id,order_date,customer_id,order_status,payment_method,device_type,order_source,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,Revenue
0,15201,Hai Phong,East,District #13,5280.0,2012-07-25,1.0,delivered,cod,desktop,paid_search,475.0,1.0,12627.39,0.0,NaN,NaN,12627.39
1,15201,Hai Phong,East,District #13,9294.0,2012-08-09,5.0,delivered,credit_card,desktop,paid_search,2046.0,5.0,5250.12,0.0,NaN,NaN,26250.60
2,15201,Hai Phong,East,District #13,13047.0,2012-08-27,3.0,delivered,credit_card,mobile,social_media,2054.0,1.0,7169.36,0.0,NaN,NaN,7169.36
3,15201,Hai Phong,East,District #13,31032.0,2012-11-26,5.0,delivered,cod,tablet,social_media,2252.0,2.0,1594.74,0.0,NaN,NaN,3189.48
4,15201,Hai Phong,East,District #13,31033.0,2012-11-24,3.0,delivered,apple_pay,desktop,social_media,2253.0,3.0,1610.73,0.0,NaN,NaN,4832.19


In [121]:
max_rev= df_final_merge.groupby('region')['Revenue'].sum().sort_values(ascending=False)
print("Khu vực có doanh thu cao nhất:", max_rev.index[0])

Khu vực có doanh thu cao nhất: East


### ==> Đáp án: C

## Question 8:

In [122]:
df_orders_cancelled= df_orders[df_orders['order_status']== 'cancelled']
df_q8= df_orders_cancelled.groupby('order_status')['payment_method'].value_counts()
print("Phương thức thanh toán phổ biến nhất cho đơn hàng bị hủy:", df_q8.index[0][1])

Phương thức thanh toán phổ biến nhất cho đơn hàng bị hủy: credit_card


### ==> Đáp án: A

## Question 9:

In [124]:
returns_with_size = df_returns.merge(df_products[['product_id', 'size']], on='product_id', how='left')
return_count_by_size = returns_with_size.groupby('size').size()

In [125]:
items_with_size = df_order_items.merge(df_products[['product_id', 'size']], on='product_id', how='left')
total_items_by_size = items_with_size.groupby('size').size()

In [126]:
return_rate = return_count_by_size / total_items_by_size
print("Tỷ lệ trả hàng theo kích thước:")
print(return_rate.sort_values(ascending=False))

Tỷ lệ trả hàng theo kích thước:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64


### ==> Đáp án: A

## Question 10:

In [129]:
df_payment= pd.read_csv('../dataset/payments.csv')
df_payment.head()

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


In [131]:
avg_payments= df_payment.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print("Giá trị thanh toán trung bình theo số lần trả góp:")
print(avg_payments)

Giá trị thanh toán trung bình theo số lần trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64


### ==> Đáp án: C